# Primary Tumor Type by MSI Status (Supplementary Table S4)

Produces **Table S4**: for the same ICI-treated cohort as Table 1
(`11_demographic_table.ipynb`) and Table S2, cross-tabulates primary
tumor type against MSI status (MSI vs MSS) — a full disclosure of exactly
how many MSI-high and microsatellite-stable patients make up each tumor
type, since several analyses in this project (the MSI-status supplementary
response figure, `18_msi_response.ipynb`; tumor-type-stratified models
that don't separately adjust for MSI) group or compare across tumor types
without always showing this breakdown explicitly.

Tumor Type rows are nested exactly as in Table S2: the five broad
`cohort_broad` categories (SKCM/NSCLC/BLCA/CRC-MSI/**Other**), where
"Other" is a subtotal row spanning every patient in that bucket, followed
immediately by a granular breakdown of "Other" into its individual
primary tumor types. A granular type gets its own row once it has at
least `SUBCOHORT_MIN_OTHER_TYPE_N` (`helpers/config.r`) ICI-treated
patients; smaller types are pooled into a residual **"Other (Other)"**
row. A final "All tumor types" row gives the grand total.

MSI Status columns are **MSI** and **MSS**, derived from `msi_high`
(`0_prepare/3_meta.ipynb`'s `as.integer(msStatus == "MSI")`) — the same
field used for Table 1's "MSI-H" row and for the MSI-status grouping in
`18_msi_response.ipynb`. A "Total" column gives the row's known-MSI-status
patient count.

**Note on `cohort_pan`'s "Non-CRC MSI" category:** `assign_pan_cohorts()`
(`3_meta.ipynb`) deliberately pools every MSI-high patient outside
colorectal cancer into a single cross-tissue "Non-CRC MSI" `cohort_pan`
value, for the pan-cancer landscape figure. That pooling is the wrong
granularity for this table: Table S4 already has an explicit MSI/MSS
column, so a "Non-CRC MSI (Other)" row would hide these patients' real
primary tumor type behind information the columns already show, and would
effectively encode MSI status on both axes at once. Before building the
"Other"-bucket breakdown, these patients are therefore reclassified by
tissue alone (same tissue-matching rules as `assign_pan_cohorts()`, minus
the colorectal/MSI branches — those can never apply here, since a
colorectal MSI-high patient is always assigned `cohort_pan == "CRC-MSI"`,
never `"Non-CRC MSI"`) so they appear under their actual primary tumor
type — e.g. an MSI-high breast cancer patient appears as "Breast cancer
(BRCA) (Other)" with MSI recorded in the MSI column, not as a separate
"Non-CRC MSI (Other)" row.

Each interior cell reports `n (row %)`, i.e. within each Tumor Type row,
the percentage of known-MSI-status patients that are MSI vs MSS. Patients
with unknown MSI status (`msi_high` is `NA`) are excluded, consistent with
`18_msi_response.ipynb`'s handling of unknown MSI status; the excluded
count is reported in a footnote. The "Total" column and "All tumor types"
row are plain counts (no percentage), since they are margins rather than
within-row distributions. As with Table S2, Other-bucket patients are
counted in both the "Other" subtotal row and their own granular child
row — these are not meant to be summed together; the "All tumor types"
grand total is computed independently from one row per patient.

**Input:** `cpi_ready.rds`
**Output:**
`figs/table_s4_tumor_type_by_msi_status.csv` -- kept purely as the intermediate
`assemble_supplementary_tables_excel.R` reads to build the combined
`supplementary_tables.xlsx`; no standalone `.html`/`.docx` are generated for
this table (see the combined workbook instead).

In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))  # tidyverse + fct_pin_last() + full_cohort_name()
source(paste0(dirname(getwd()), "/helpers/config.r"))

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

In [ ]:
# ── Load ICI-ready data ───────────────────────────────────────────────────────
# Same population as Table 1 and Table S2: ICI-treated patients with known
# RECIST response, one row per patient.
#
# IMPORTANT: cpi_ready.rds stores one row per (sampleId, gp) *and* a
# duplicated copy of every row with cohort_broad overwritten to
# "ICI-Treated" (added in 4_base.ipynb so Figure 3 can facet pooled-vs-per-
# cancer-type together). The gp filter below removes the first
# duplication; distinct() removes the second.
df_raw <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds")) |>
  filter(gp == "T-cell", !is.na(response)) |>
  distinct(sampleId, .keep_all = TRUE)

In [ ]:
# ── Variable preparation ──────────────────────────────────────────────────────
# Tumor Type rows: same broad/nested design as Table S2 -- the five
# cohort_broad categories plus a nested breakdown of "Other" into its
# granular primary tumor types (>= SUBCOHORT_MIN_OTHER_TYPE_N patients get
# their own row; smaller types pool into "Other (Other)").
#
# MSI Status: msi_high is numeric 1/0/NA (same field as Table 1's "MSI-H"
# row and 18_msi_response.ipynb's MSI grouping). NA means unknown MSI
# status, not MSS -- handled as a separate exclusion below, not folded
# into "MSS".
demo <- df_raw |>
  mutate(
    # Full tumor-type names (not bare acronyms) for this supplemental
    # table -- see full_cohort_name() in helpers/shortcuts.r.
    tumor_type_broad = fct_pin_last(factor(full_cohort_name(cohort_broad)), "Other"),
    msi_status        = factor(
      ifelse(msi_high == 1, "MSI", "MSS"),
      levels = c("MSI", "MSS")
    )
  )

# ── MSI-agnostic tissue reclassification for "Non-CRC MSI" patients ───────────
# cohort_pan (assign_pan_cohorts(), 3_meta.ipynb) pools every MSI-high
# patient outside colorectal cancer into a single cross-tissue "Non-CRC
# MSI" category, for the pan-cancer landscape figure. That's the wrong
# granularity here: this table already has an explicit MSI/MSS column, so
# a pooled "Non-CRC MSI (Other)" row would hide these patients' real
# primary tumor type behind information already shown in the columns.
# Reclassify them by tissue alone before building the "Other" breakdown,
# using the same tissue-matching rules as assign_pan_cohorts() with the
# colorectal/MSI branches omitted -- those can never apply here, since a
# colorectal MSI-high patient is always cohort_pan == "CRC-MSI", never
# "Non-CRC MSI" (assign_pan_cohorts() checks colon/rectum + MSI-high
# before the Non-CRC MSI catch-all).
reclassify_non_crc_msi <- function(location, type) {
  case_when(
    location == "Breast" ~ "BRCA",
    location == "Lung" & grepl("Non-small cell|Other|Unknown", type) ~ "NSCLC",
    location == "Lung" & grepl("Small cell|Neuroendocrine|carcinoid", type) ~ "LNET",
    location == "Prostate" ~ "PRAD",
    location == "Soft tissue" ~ "SARC",
    location == "Skin" ~ "SKCM",
    location == "Ovary" ~ "OV",
    location == "Brain/central nervous system" ~ "CNS",
    location == "Pancreas" ~ "PAAD",
    location %in% c("Kidney", "Ureter/renal pelvis") ~ "KIRC",
    location == "Esophagus/gastroesophageal junction" ~ "ESOG",
    location %in% c("Bladder", "Urethra") ~ "BLCA",
    location == "Biliary tract" ~ "CHOL",
    location == "Stomach" ~ "STAD",
    location == "Cervix" ~ "CESC",
    location == "Endometrium/uterus" ~ "ENDO",
    location %in% c("Pleura", "Peritoneum") | type == "Mesothelioma" ~ "MESO",
    location == "Liver" ~ "LIHC",
    location == "Cancer of unknown primary (CUP)" ~ "CUP",
    TRUE ~ "Other"
  )
}

demo <- demo |>
  mutate(
    cohort_pan_tissue = ifelse(
      cohort_pan == "Non-CRC MSI",
      reclassify_non_crc_msi(primaryTumorLocation, primaryTumorType),
      as.character(cohort_pan)
    )
  )

n_reclassified <- sum(demo$cohort_pan == "Non-CRC MSI", na.rm = TRUE)
if (n_reclassified > 0) {
  message(sprintf(
    paste0(
      "Reclassified %d \"Non-CRC MSI\" patient(s) into their real primary tumor ",
      "type for Table S4's tumor-type axis (MSI status is already captured by ",
      "the MSI/MSS columns)."
    ),
    n_reclassified
  ))
}

other_subtype_counts <- demo |>
  filter(cohort_broad == "Other") |>
  count(cohort_pan_tissue, name = "n")

well_represented_other_types <- other_subtype_counts |>
  filter(n >= SUBCOHORT_MIN_OTHER_TYPE_N) |>
  pull(cohort_pan_tissue)

demo <- demo |>
  mutate(
    tumor_type_other_detail = case_when(
      cohort_broad != "Other" ~ NA_character_,
      cohort_pan_tissue %in% well_represented_other_types ~
        paste0(full_cohort_name(cohort_pan_tissue), " (Other)"),
      TRUE ~ "Other (Other)"
    ),
    tumor_type_other_detail = fct_pin_last(factor(tumor_type_other_detail), "Other (Other)")
  )

message("\"Other\" bucket breakdown (types with N >= ", SUBCOHORT_MIN_OTHER_TYPE_N, " shown individually):")
print(levels(demo$tumor_type_other_detail))

In [ ]:
# ── Build cross-tab: Tumor Type x MSI Status, n (row %) per cell ──────────────
# Rows: the five broad cohorts (SKCM/NSCLC/BLCA/CRC-MSI/"Other" as a
# subtotal), followed by a nested breakdown of "Other", followed by an
# "All tumor types" grand total -- same subtotal-plus-detail-rows design as
# Table S2 (see Table S2's cell-4 for the full rationale).
#
# Interior cells report "n (row %)": the percentage of that Tumor Type's
# known-MSI-status patients that are MSI vs MSS. The "Total" column and
# "All tumor types" row are plain counts (table margins), not row
# percentages of themselves.
#
# Tumor Type is always known by construction (cohort_broad's case_when()
# falls back to "Other"); MSI Status can be genuinely unknown (msi_high is
# NA when msStatus wasn't called/available) -- that is the only source of
# exclusion here, diagnosed below.
known <- demo |> filter(!is.na(msi_status))

n_total    <- nrow(demo)
n_known    <- nrow(known)
n_excluded <- n_total - n_known
if (n_excluded > 0) {
  message(sprintf(
    "Note: %d of %d ICI-treated patients excluded from Table S4 (unknown MSI status).",
    n_excluded, n_total
  ))
}

summarise_rows <- function(df, type_col) {
  df |>
    filter(!is.na({{ type_col }})) |>
    group_by(`Tumor Type` = {{ type_col }}, `MSI Status` = msi_status) |>
    summarise(n = n(), .groups = "drop") |>
    group_by(`Tumor Type`) |>
    mutate(label = sprintf("%d (%d%%)", n, round(100 * n / sum(n)))) |>
    ungroup() |>
    select(`Tumor Type`, `MSI Status`, label)
}

row_margins <- function(df, type_col) {
  df |>
    filter(!is.na({{ type_col }})) |>
    group_by(`Tumor Type` = {{ type_col }}) |>
    summarise(n = n(), .groups = "drop") |>
    mutate(`MSI Status` = "Total", label = as.character(n)) |>
    select(`Tumor Type`, `MSI Status`, label)
}

# Column margins and the grand total use `known` directly (one row per
# patient) so Other-bucket patients are counted once here, not twice --
# same reasoning as Table S2's col_totals/grand_total.
other_known <- known |> filter(cohort_broad == "Other")

col_margins <- known |>
  group_by(`MSI Status` = msi_status) |>
  summarise(n = n(), .groups = "drop") |>
  mutate(`Tumor Type` = "All tumor types", label = as.character(n)) |>
  select(`Tumor Type`, `MSI Status`, label)

grand_total <- tibble(
  `Tumor Type` = "All tumor types", `MSI Status` = "Total",
  label = as.character(nrow(known))
)

cell_summary <- bind_rows(
  summarise_rows(known,       tumor_type_broad),
  summarise_rows(other_known, tumor_type_other_detail)
)

row_totals <- bind_rows(
  row_margins(known,       tumor_type_broad),
  row_margins(other_known, tumor_type_other_detail)
)

# Row order: broad cohorts (freq order, "Other" pinned last as a
# subtotal), immediately followed by its granular children (freq order,
# "Other (Other)" pinned last), then the grand total. Column order:
# MSI, MSS, then the row-total column.
type_order <- c(
  setdiff(levels(demo$tumor_type_broad), "Other"),
  "Other",
  levels(demo$tumor_type_other_detail),
  "All tumor types"
)
msi_order <- c("MSI", "MSS", "Total")

# Split out of the main pipe since the final column-reordering step needs
# to inspect the pivoted frame's own column names -- the native R pipe
# (|>) doesn't support referring to the LHS from inside a nested call the
# way magrittr's %>% does (same issue documented in Table S2's cell-4).
wide_pivoted <- bind_rows(cell_summary, row_totals, col_margins, grand_total) |>
  mutate(`Tumor Type` = factor(`Tumor Type`, levels = type_order)) |>
  arrange(`Tumor Type`) |>
  tidyr::pivot_wider(names_from = `MSI Status`, values_from = label, values_fill = "\u2014")

wide <- wide_pivoted |>
  select(`Tumor Type`, all_of(intersect(msi_order, names(wide_pivoted))))

wide

In [ ]:
# ── Save outputs ──────────────────────────────────────────────────────────────
out_dir <- paste0(CIDER_PROJ_DIR, "figs/")

footnote_text <- if (n_excluded > 0) {
  sprintf(
    paste0(
      "%d of %d ICI-treated patients (Table 1's cohort) are excluded here due to ",
      "unknown MSI status. Interior cells report n (%% of that Tumor Type's ",
      "known-MSI-status patients); \"Total\"/\"All tumor types\" are plain counts. ",
      "\"Other\" is a subtotal spanning the rows below it; row/column totals should ",
      "not be summed with it."
    ),
    n_excluded, n_total
  )
} else {
  sprintf(
    paste0(
      "All %d ICI-treated patients (Table 1's cohort) are included. Interior ",
      "cells report n (%% of that Tumor Type's known-MSI-status patients); ",
      "\"Total\"/\"All tumor types\" are plain counts. \"Other\" is a subtotal ",
      "spanning the rows below it; row/column totals should not be summed with it."
    ),
    n_total
  )
}

# CSV — build intermediate only. Not a standalone deliverable: this
# table's actual output is its sheet in the combined
# supplementary_tables.xlsx (assemble_supplementary_tables_excel.R reads
# this .csv + the metadata sidecar below to build it).
wide |>
  readr::write_csv(paste0(out_dir, "table_s4_tumor_type_by_msi_status.csv"))

message("Saved: table_s4_tumor_type_by_msi_status.csv")

# Metadata sidecar (title + footnote) -- consumed by
# assemble_supplementary_tables_excel.R to build the combined supplemental
# tables Excel workbook without re-deriving this table's title/footnote
# text (footnote_text is computed above).
jsonlite::write_json(
  list(
    title    = "Supplementary Table S4. Primary tumor type by MSI status",
    footnote = footnote_text
  ),
  paste0(out_dir, "table_s4_tumor_type_by_msi_status_meta.json"),
  auto_unbox = TRUE
)